# 15 · Datenanalyse mit pandas

**Ausblick:** Dieses Notebook ist eine Vorführung ohne Übungen. Es zeigt, wie Datenanalyse im Berufsalltag aussieht, mit der Bibliothek **pandas**.

**Lernziele**

- Du liest CSV- und Excel-Dateien in eine Tabelle (`DataFrame`) ein.
- Du verschaffst dir einen ersten Überblick: `head`, `info`, `describe`.
- Du wählst Spalten aus, filterst und sortierst Zeilen.
- Du zählst, gruppierst und berechnest neue Spalten.
- Du zeichnest Diagramme direkt aus der Tabelle und schreibst Ergebnisse zurück nach Excel.

Alles, was wir hier zeigen, ginge auch mit Listen, Dictionaries und Schleifen aus den Notebooks 04 bis 07. pandas erledigt dieselben Schritte in einer Zeile und rechnet auch mit Millionen Zeilen schnell. Gegenüber Excel hat Code einen großen Vorteil: Kommt nächste Woche eine neue Datei, läuft dieselbe Auswertung auf Knopfdruck noch einmal.

`pandas` und `matplotlib` (für Diagramme) sind keine Teile der Standardbibliothek. Sie stehen in `pyproject.toml` und kommen mit `uv sync` in die Umgebung. Üblich ist die Abkürzung `pd`:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

## Die Daten: ein Schweißprotokoll

Ein Schweißroboter verbindet Metallteile mit einer Naht. Für jede Naht schreibt er zwei Einstellungen mit; danach prüft die Qualitätssicherung, ob die Naht hält.

| Spalte | bedeutet |
| --- | --- |
| `Zeitpunkt` | wann die Naht geschweißt wurde |
| `Naht` | Nummer der Naht |
| `Roboter` | welcher der drei Roboter geschweißt hat |
| `Strom_A` | Stromstärke in Ampere: je mehr Strom, desto heißer wird die Naht |
| `Geschwindigkeit_cm_min` | wie viele Zentimeter Naht der Roboter pro Minute schafft |
| `Ergebnis` | Prüfung bestanden (`ok`) oder nicht (`nicht ok`) |

Die Faustregel: Zu viel Hitze bei langsamer Fahrt brennt durch, zu wenig Hitze bei schneller Fahrt hält nicht. Drei Roboter (R1 bis R3) arbeiten drei Wochen lang. Die Werte sind simuliert, erzeugt mit `tools/make_ausblick_data.py`.

So sehen die ersten Zeilen der Datei aus:

In [ ]:
from pathlib import Path

lines = Path("../daten/schweissprotokoll.csv").read_text(encoding="utf-8").splitlines()
for line in lines[:3]:
    print(line)

## CSV einlesen

`pd.read_csv()` liest eine CSV-Datei in einen **DataFrame**: eine Tabelle mit benannten Spalten, ähnlich einem Excel-Blatt. Ohne weitere Angaben erwartet pandas Kommas als Trennzeichen:

In [ ]:
wrong = pd.read_csv("../daten/schweissprotokoll.csv")
wrong.shape  # (Zeilen, Spalten)

600 Zeilen, aber nur **eine** Spalte: Deutsche Programme trennen mit Semikolon, weil das Komma schon als Dezimalzeichen dient. Mit `sep` und `decimal` sagen wir pandas, wie die Datei aufgebaut ist; `parse_dates` macht aus dem Text in `Zeitpunkt` echte Zeitpunkte wie in Notebook 11:

In [ ]:
log = pd.read_csv(
    "../daten/schweissprotokoll.csv",
    sep=";",
    decimal=",",
    parse_dates=["Zeitpunkt"],
    date_format="%d.%m.%Y %H:%M",
)
log.head()

## Erster Überblick

Drei Befehle stehen am Anfang fast jeder Analyse:

| Befehl | zeigt |
| --- | --- |
| `head()` | die ersten 5 Zeilen (`tail()` die letzten) |
| `info()` | Spalten, Datentypen, fehlende Werte |
| `describe()` | Kennzahlen jeder Zahlenspalte: Anzahl, Mittelwert, Minimum, Maximum, Quartile |

`info()` verrät, ob das Einlesen geklappt hat: Zahlen müssen als `float64` oder `int64` ankommen, nicht als Text.

In [ ]:
log.info()

In [ ]:
log.describe().round(1)

## Spalten auswählen

Eine Spalte holst du wie bei einem Dictionary mit eckigen Klammern. Heraus kommt eine **Series**: eine einzelne Spalte mit Zeilennummern. Rechnungen wie `mean()` oder `max()` laufen über die ganze Spalte, ohne Schleife:

In [ ]:
current = log["Strom_A"]
print(current.mean(), current.min(), current.max())

Mehrere Spalten wählst du mit einer **Liste** von Namen aus, daher die doppelten Klammern:

In [ ]:
log[["Naht", "Roboter", "Strom_A"]].head(3)

## Zeilen filtern und sortieren

Ein Vergleich mit einer Spalte liefert für jede Zeile `True` oder `False`. Steckst du das Ergebnis in eckige Klammern, bleiben nur die Zeilen mit `True` übrig:

In [ ]:
failed = log[log["Ergebnis"] == "nicht ok"]
len(failed)

Mehrere Bedingungen verknüpfst du mit `&` (und) und `|` (oder), jede Bedingung in runden Klammern. `and` und `or` aus Notebook 03 funktionieren hier nicht, weil sie nur einen einzelnen Wahrheitswert prüfen, nicht eine ganze Spalte:

In [ ]:
log[(log["Roboter"] == "R3") & (log["Strom_A"] > 260)].head()

`sort_values()` sortiert nach einer Spalte, mit `ascending=False` absteigend. Die drei Nähte mit dem höchsten Strom:

In [ ]:
log.sort_values("Strom_A", ascending=False).head(3)

## Zählen und neue Spalten

`value_counts()` zählt, wie oft jeder Wert vorkommt; mit `normalize=True` als Anteil:

In [ ]:
log["Ergebnis"].value_counts(normalize=True).round(2)

Eine neue Spalte legst du an, indem du ihr einen Wert zuweist, wie einem neuen Schlüssel im Dictionary. Die Rechnung läuft für alle 600 Zeilen auf einmal.

Die Spalte `Fehler` enthält `True` oder `False`; mit ihr rechnen wir gleich die Fehlerquote aus. Für die Schicht schreiben wir eine eigene Funktion wie in Notebook 07; `apply()` ruft sie für jeden Wert der Spalte auf. `dt.hour` holt die Stunde aus dem Zeitpunkt:

In [ ]:
def shift(hour):
    if 6 <= hour < 14:
        return "Früh"
    elif 14 <= hour < 22:
        return "Spät"
    else:
        return "Nacht"


log["Fehler"] = log["Ergebnis"] == "nicht ok"
log["Schicht"] = log["Zeitpunkt"].dt.hour.apply(shift)
log[["Zeitpunkt", "Schicht", "Ergebnis", "Fehler"]].head()

## Gruppieren

`groupby()` teilt die Tabelle nach den Werten einer Spalte in Gruppen und rechnet für jede Gruppe getrennt, wie eine Pivot-Tabelle in Excel. Der Mittelwert einer `True`/`False`-Spalte ist der Anteil der `True`-Werte, hier also die Fehlerquote:

In [ ]:
log.groupby("Roboter")["Fehler"].mean().round(2)

R3 fällt auf. Mit `agg()` berechnest du mehrere Kennzahlen auf einmal; jede bekommt einen eigenen Namen:

In [ ]:
log.groupby("Roboter").agg(
    Naehte=("Naht", "count"),
    Strom=("Strom_A", "mean"),
    Fehlerquote=("Fehler", "mean"),
).round(2)

Hat R3 schon immer mit mehr Strom geschweißt? `pivot_table()` legt die Kalenderwoche in die Zeilen und die Roboter in die Spalten:

In [ ]:
log["Woche"] = log["Zeitpunkt"].dt.isocalendar().week
weekly = log.pivot_table(index="Woche", columns="Roboter", values="Strom_A", aggfunc="mean")
weekly.round(1)

## Diagramme

Jede Tabelle und jede Spalte hat eine Methode `plot()`; dahinter zeichnet matplotlib. Als Linie wird die Drift von R3 sofort sichtbar:

In [ ]:
weekly.plot(marker="o", title="Mittlerer Strom je Woche", ylabel="Strom in A", xticks=weekly.index)
plt.show()

Ein Streudiagramm zeigt jede Naht als Punkt, gefärbt nach dem Prüfergebnis. Die fehlerhaften Nähte (rot) liegen genau dort, wo die Faustregel es sagt: oben links viel Strom bei langsamer Fahrt, unten rechts wenig Strom bei schneller Fahrt. Genau solche Muster lernt ein Modell im Notebook zum maschinellen Lernen.

In [ ]:
colors = log["Fehler"].map({True: "red", False: "green"})
log.plot.scatter(x="Geschwindigkeit_cm_min", y="Strom_A", c=colors, alpha=0.6)
plt.show()

## Excel einlesen

`pd.read_excel()` liest ein Tabellenblatt einer `.xlsx`-Datei. Unsere Datei ist ein Umsatzbericht, wie er aus einer Fachabteilung kommt: Titel und Stand stehen über der Tabelle, eine Gesamtzeile darunter. Liest du sie ohne weitere Angaben, hält pandas die Titelzeile für die Spaltennamen:

In [ ]:
pd.read_excel("../daten/umsatz_filialen.xlsx").head()

Die Spaltennamen stehen in Excel-Zeile 4. pandas zählt ab 0, also `header=3`. Am Ende hängen noch die Leerzeile und die Gesamtzeile:

In [ ]:
sales = pd.read_excel("../daten/umsatz_filialen.xlsx", header=3)
sales.tail(3)

Nur echte Filialen haben eine Region. `dropna(subset=[...])` wirft alle Zeilen weg, in denen diese Spalte leer ist (`NaN`, "not a number", ist pandas' Wert für eine leere Zelle):

In [ ]:
sales = sales.dropna(subset=["Region"])
sales.groupby("Region")["Summe"].sum()

## Zwei Tabellen verbinden

Das zweite Blatt `Ziele` enthält das Jahresziel jeder Filiale. `sheet_name` wählt das Blatt, `merge()` verbindet beide Tabellen über die gemeinsame Spalte `Filiale`, wie `SVERWEIS` in Excel:

In [ ]:
targets = pd.read_excel("../daten/umsatz_filialen.xlsx", sheet_name="Ziele")
report = sales[["Filiale", "Region", "Summe"]].merge(targets, on="Filiale")
report["Quote_%"] = (report["Summe"] / report["Jahresziel"] * 100).round(1)
report.sort_values("Quote_%")

Nach neun von zwölf Monaten wären 75 % im Plan; alle Filialen liegen knapp darunter, Gießen am weitesten.

Die Monatsspalten als Linien, eine je Filiale. `set_index()` macht die Filialen zu Zeilennamen, `.T` vertauscht Zeilen und Spalten, damit die Monate auf der x-Achse landen:

In [ ]:
months = ["Jan", "Feb", "Mär", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep"]
sales.set_index("Filiale")[months].T.plot(title="Umsatz je Monat", ylabel="Euro")
plt.legend(bbox_to_anchor=(1, 1))  # Legende rechts neben das Diagramm
plt.show()

## Ergebnisse speichern

`to_excel()` und `to_csv()` schreiben eine Tabelle in eine Datei; `index=False` lässt die Zeilennummern weg. Für deutsches Excel bekommt die CSV-Datei wieder Semikolon und Dezimalkomma:

In [ ]:
out_dir = Path("../daten/ausgabe")
out_dir.mkdir(exist_ok=True)
report.to_excel(out_dir / "zielerreichung.xlsx", index=False)
report.to_csv(out_dir / "zielerreichung.csv", sep=";", decimal=",", index=False)
print(sorted(path.name for path in out_dir.glob("zielerreichung.*")))

`to_excel()` schreibt nur Werte: keine Schriftarten, Spaltenbreiten oder Formeln. Wie du einzelne Zellen und Bereiche gezielt liest, formatierst und mit Formeln füllst, zeigt das nächste Notebook mit `openpyxl`.

## Zusammenfassung

- `pd.read_csv()` und `pd.read_excel()` lesen Dateien in einen `DataFrame`. Deutsche CSV-Dateien brauchen `sep=";"` und `decimal=","`; Berichte mit Kopfzeilen `header=`, mehrere Blätter `sheet_name=`.
- Überblick: `head()`, `info()`, `describe()`. Prüfe mit `info()`, ob Zahlen als Zahlen ankommen.
- `df["Spalte"]` ist eine Series, `df[["A", "B"]]` eine kleinere Tabelle. Rechnungen laufen über die ganze Spalte, ohne Schleife.
- Filtern mit `df[bedingung]`, verknüpft mit `&` und `|`; sortieren mit `sort_values()`.
- `value_counts()` zählt, `groupby().agg()` und `pivot_table()` fassen zusammen, `merge()` verbindet Tabellen.
- `plot()` zeichnet direkt aus der Tabelle; `to_excel()` und `to_csv()` speichern.